# Time Series Statistical Analysis Practice Notebook

This notebook is a hands-on companion to the Markdown file on **Time Series Statistical Analysis**.  
It demonstrates key methods for sequential and time-dependent data.

Topics covered:

1. Trend analysis  
2. Seasonal decomposition  
3. Moving average  
4. Exponential smoothing  
5. Auto-regressive models  
6. ARIMA  
7. SARIMA  
8. Stationarity tests  
9. ACF and PACF  

The notebook is designed for learning, GitHub repositories, and classroom use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

np.random.seed(42)

## Create Example Time Series

We build a synthetic monthly time series with trend, seasonality, and noise.

In [ ]:
n = 120
time = np.arange(n)
trend = 0.4 * time
seasonal = 8 * np.sin(2 * np.pi * time / 12)
noise = np.random.normal(0, 2, n)
series = 50 + trend + seasonal + noise

dates = pd.date_range(start='2015-01-01', periods=n, freq='M')
ts = pd.Series(series, index=dates)
ts.head()

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values)
plt.title('Synthetic Monthly Time Series')
plt.xlabel('Date')
plt.ylabel('Value')
plt.show()

## 1. Trend Analysis

A simple linear trend model is:

$$
x_t = \beta_0 + \beta_1 t + \epsilon_t
$$

We estimate the trend slope using a linear fit.

In [ ]:
slope, intercept = np.polyfit(time, ts.values, 1)
trend_line = intercept + slope * time

pd.DataFrame({'Intercept': [intercept], 'Slope': [slope]})

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Series')
plt.plot(ts.index, trend_line, label='Linear Trend')
plt.title('Trend Analysis')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

## 2. Seasonal Decomposition

An additive decomposition assumes:

$$
x_t = T_t + S_t + R_t
$$

where:
- \(T_t\) is trend
- \(S_t\) is seasonality
- \(R_t\) is residual

In [ ]:
decomp = seasonal_decompose(ts, model='additive', period=12)
decomp.trend.head()

In [ ]:
decomp.plot()
plt.show()

## 3. Moving Average

A simple moving average with window \(k\) is:

$$
MA_t = \frac{1}{k}\sum_{i=0}^{k-1} x_{t-i}
$$

In [ ]:
ma_6 = ts.rolling(window=6).mean()
ma_12 = ts.rolling(window=12).mean()

plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Original')
plt.plot(ts.index, ma_6, label='MA 6')
plt.plot(ts.index, ma_12, label='MA 12')
plt.title('Moving Averages')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

## 4. Exponential Smoothing

Simple exponential smoothing is:

$$
\hat{x}_{t+1}=\alpha x_t + (1-\alpha)\hat{x}_t
$$

In [ ]:
ses_model = SimpleExpSmoothing(ts).fit()
ses_fitted = ses_model.fittedvalues

plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Original')
plt.plot(ts.index, ses_fitted, label='Simple Exp Smoothing')
plt.title('Simple Exponential Smoothing')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

### Holt-Winters Exponential Smoothing

This version includes trend and seasonality.

In [ ]:
hw_model = ExponentialSmoothing(ts, trend='add', seasonal='add', seasonal_periods=12).fit()
hw_fitted = hw_model.fittedvalues

plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Original')
plt.plot(ts.index, hw_fitted, label='Holt-Winters')
plt.title('Holt-Winters Exponential Smoothing')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

## 5. Auto-Regressive Model

An AR(p) model is:

$$
x_t = c + \sum_{i=1}^{p}\phi_i x_{t-i} + \epsilon_t
$$

In [ ]:
ar_model = AutoReg(ts, lags=3, old_names=False).fit()
ar_pred = ar_model.predict(start=3, end=len(ts)-1)
ar_model.params

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Original')
plt.plot(ar_pred.index, ar_pred.values, label='AR(3) Fitted')
plt.title('Auto-Regressive Model')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

## 6. ARIMA

ARIMA combines autoregression, differencing, and moving-average errors.

Here we fit a simple ARIMA(2,1,2) model.

In [ ]:
arima_model = ARIMA(ts, order=(2, 1, 2)).fit()
arima_forecast = arima_model.forecast(steps=12)
arima_model.summary().tables[1]

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Original')
future_idx = pd.date_range(ts.index[-1] + pd.offsets.MonthEnd(), periods=12, freq='M')
plt.plot(future_idx, arima_forecast.values, label='ARIMA Forecast')
plt.title('ARIMA Forecast')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

## 7. SARIMA

SARIMA extends ARIMA with seasonal terms.

We fit a simple seasonal model with seasonal period 12.

In [ ]:
sarima_model = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12)).fit(disp=False)
sarima_forecast = sarima_model.forecast(steps=12)
sarima_model.summary().tables[1]

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts.index, ts.values, label='Original')
plt.plot(future_idx, sarima_forecast.values, label='SARIMA Forecast')
plt.title('SARIMA Forecast')
plt.xlabel('Date')
plt.ylabel('Value')
plt.legend()
plt.show()

## 8. Stationarity Test

Most classical time-series models assume stationarity.

We use the **Augmented Dickey-Fuller (ADF)** test.

Null hypothesis:
- \(H_0\): the series has a unit root and is non-stationary

Alternative:
- \(H_1\): the series is stationary

In [ ]:
adf_result = adfuller(ts)
adf_summary = pd.DataFrame({
    'Metric': ['ADF Statistic', 'p Value', 'Used Lags', 'Number of Observations'],
    'Value': [adf_result[0], adf_result[1], adf_result[2], adf_result[3]]
})
adf_summary

### First Difference

If the original series is non-stationary, differencing is a common remedy.

In [ ]:
ts_diff = ts.diff().dropna()
adf_diff = adfuller(ts_diff)
pd.DataFrame({
    'Metric': ['ADF Statistic (Differenced)', 'p Value (Differenced)'],
    'Value': [adf_diff[0], adf_diff[1]]
})

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(ts_diff.index, ts_diff.values)
plt.title('First-Differenced Series')
plt.xlabel('Date')
plt.ylabel('Differenced Value')
plt.show()

## 9. ACF and PACF

ACF and PACF help inspect lag dependence and guide ARIMA model order selection.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(ts_diff, lags=24, ax=axes[0])
plot_pacf(ts_diff, lags=24, ax=axes[1], method='ywm')
axes[0].set_title('ACF of Differenced Series')
axes[1].set_title('PACF of Differenced Series')
plt.tight_layout()
plt.show()

## 10. Summary Table

This table gathers some key outputs from the time-series analysis.

In [ ]:
summary = pd.DataFrame({
    'Measure': [
        'Trend slope',
        'ADF p-value (original)',
        'ADF p-value (differenced)',
        'AR(3) intercept',
        'ARIMA AIC',
        'SARIMA AIC'
    ],
    'Value': [
        slope,
        adf_result[1],
        adf_diff[1],
        ar_model.params.iloc[0] if hasattr(ar_model.params, 'iloc') else ar_model.params[0],
        arima_model.aic,
        sarima_model.aic
    ]
})
summary

## 11. Mini Exercises

Try these on your own:

1. Change the moving-average window and compare smoothing behavior.  
2. Change the seasonal period from 12 to another value and inspect decomposition.  
3. Fit different AR orders and compare AIC.  
4. Try different ARIMA and SARIMA orders.  
5. Apply the ADF test to your own convergence or sensor series.  
6. Replace the synthetic data with building energy, SHM, or optimizer-history data.

These exercises are especially useful for AI, machine learning, structural health monitoring, forecasting, and digital-twin applications.